# T3 준비 — FinRL 포트폴리오 배분(ICAIF 2021) 데이터 고정과 학습 1회 시간 측정

**사전 등록:** `notes/prereg_reeval_v3.md`(6f8f512). 이 노트북은 v3 9장의 계산 예산을 채우려고 시간만 잰다. **테스트 성과는 계산하지 않는다.** 학습된 시드 0 모델은 Drive에 저장해 두고, O1 노트북에서 그대로 쓴다.

**설정(v3 3장).**
- 기간: 논문 Table 3. 학습 2009-01-01~2020-06-30, 거래 2020-07-01~2021-06-30. 데이터는 2009-01-01~2021-07-01을 받는다(튜토리얼처럼 학습 시작일부터 받고, 공분산 lookback 252일).
- 하이퍼파라미터·timesteps: FinRL-Tutorials `1-Introduction/FinRL_PortfolioAllocation_NeurIPS_2020.py`
  - A2C: n_steps 5, ent 0.005, lr 2e-4, 50k
  - PPO: n_steps 2048, ent 0.005, lr 1e-4, batch 128, 80k
  - DDPG: batch 128, buffer 5e4, lr 1e-3, 50k
  - TD3: batch 100, buffer 1e6, lr 1e-3, 30k
- env: FinRL-Meta `env_portfolio_yahoofinance.StockPortfolioEnv` @286fef6.

**v3 2.3 허용 수정(기록).**
1. 고정 커밋에서 튜토리얼이 그대로 돌지 않는다. `DataProcessor`에 문자열을 넘기면 소스 비교(`== DataSource.yahoofinance`)에서 실패하고, Enum을 넘기면 `run()`의 캐시 파일명 `"_".join(...)`에서 실패한다. 튜토리얼의 `dp.data_split`도 기본 열 `time`을 쓰는데, 그 직전에 열 이름을 `date`로 바꾼다. → 내부 처리기 `Yahoofinance`의 `download_data → clean_data → add_technical_indicator`를 직접 호출하고, `data_split`에는 `target_date_col='date'`를 넘긴다(같은 처리 순서).
2. 호환 shim: yfinance가 `proxy` 인자를 받지 않으면 빼고 호출한다. TA-Lib가 설치되지 않으면 빈 `talib` 모듈을 둔다(기본 지표 라이브러리는 stockstats라 쓰이지 않음).
3. SB3 2.x + gym 0.26 + shimmy: SB3가 구버전 gym env를 자동으로 감싼다.

**데이터 고정.** 처음 받은 데이터를 Drive `data/finrl_t3/snapshot.csv`에 저장한다. 이후 모든 T3 실행은 이 파일을 쓴다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pathlib
OUT = pathlib.Path('/content/drive/MyDrive/data/finrl_t3'); OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'models').mkdir(exist_ok=True)

In [ ]:
COMMIT = '286fef6'
!test -d /content/FinRL-Meta || git clone -q --filter=blob:none https://github.com/AI4Finance-Foundation/FinRL-Meta.git /content/FinRL-Meta
!cd /content/FinRL-Meta && git checkout -q $COMMIT && git log -1 --format='%H %cd'
!pip install -q gym==0.26.2 shimmy stable-baselines3 stockstats yfinance pandas_market_calendars exchange_calendars PyPortfolioOpt
!pip install -q TA-Lib || echo 'TA-Lib 설치 실패 — 빈 모듈로 대체'

In [ ]:
import sys, os, types, inspect, json, time, platform
os.chdir('/content/FinRL-Meta'); sys.path.insert(0, '/content/FinRL-Meta')
try:
    import talib
except ImportError:
    sys.modules['talib'] = types.ModuleType('talib'); print('talib: 빈 모듈 사용')
import yfinance as yf
if 'proxy' not in inspect.signature(yf.download).parameters:
    _dl = yf.download
    def _download(*a, proxy=None, **kw): return _dl(*a, **kw)
    yf.download = _download; print('yfinance: proxy 인자 제거 shim')
import numpy as np, pandas as pd, torch, stable_baselines3, gym
from meta import config, config_tickers
from meta.data_processors._base import DataSource
from meta.data_processors.yahoofinance import Yahoofinance
from meta.data_processor import DataProcessor
from meta.env_portfolio_allocation.env_portfolio_yahoofinance import StockPortfolioEnv
from agents.stablebaselines3_models import DRLAgent
for d in [config.DATA_SAVE_DIR, config.TRAINED_MODEL_DIR, config.TENSORBOARD_LOG_DIR, config.RESULTS_DIR]: os.makedirs(d, exist_ok=True)
ENV = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, torch=torch.__version__,
           sb3=stable_baselines3.__version__, gym=gym.__version__, yfinance=yf.__version__, cuda=torch.cuda.is_available(), cpus=os.cpu_count())
print(ENV); (OUT / 'env.json').write_text(json.dumps(ENV, indent=1))

In [ ]:
# 데이터 — 스냅샷이 있으면 그대로 쓴다
snap = OUT / 'snapshot.csv'
if snap.exists():
    raw = pd.read_csv(snap); print('스냅샷 사용', raw.shape)
else:
    p = Yahoofinance(DataSource.yahoofinance, '2009-01-01', '2021-07-01', '1d')  # clean_data는 소문자 '1d'만 지원
    p.download_data(config_tickers.DOW_30_TICKER)
    p.clean_data()
    p.add_technical_indicator(config.INDICATORS)
    raw = p.dataframe.copy(); raw.to_csv(snap, index=False); print('스냅샷 저장', raw.shape)
df = raw.rename(columns={'time': 'date'})
df = df.sort_values(['date', 'tic'], ignore_index=True)
df.index = df.date.factorize()[0]
if 'index' in df.columns: df = df.drop(columns=['index'])
# 튜토리얼 그대로: 252일 공분산·수익 창
cov_list, return_list, lookback = [], [], 252
for i in range(lookback, len(df.index.unique())):
    data_lookback = df.loc[i - lookback:i, :]
    price_lookback = data_lookback.pivot_table(index='date', columns='tic', values='close').dropna(axis=1)
    return_lookback = price_lookback.pct_change().dropna()
    return_list.append(return_lookback); cov_list.append(return_lookback.cov().values)
df_cov = pd.DataFrame({'date': df.date.unique()[lookback:], 'cov_list': cov_list, 'return_list': return_list})
df = df.merge(df_cov, on='date').sort_values(['date', 'tic']).reset_index(drop=True)
split = lambda s, e: DataProcessor.data_split(None, df, s, e, target_date_col='date')
train = split('2009-01-01', '2020-07-01')
stock_dimension = len(train.tic.unique())
print('종목 수', stock_dimension, sorted(train.tic.unique()))
print('학습 기간', train.date.min(), '~', train.date.max(), '일수', train.index.nunique())
env_kwargs = {'hmax': 100, 'initial_amount': 1000000, 'transaction_cost_pct': 0.001, 'state_space': stock_dimension,
              'stock_dim': stock_dimension, 'tech_indicator_list': config.INDICATORS, 'action_space': stock_dimension, 'reward_scaling': 1e-4}

In [ ]:
%%time
PARAMS = {'a2c': ({'n_steps': 5, 'ent_coef': 0.005, 'learning_rate': 0.0002}, 50000),
          'ppo': ({'n_steps': 2048, 'ent_coef': 0.005, 'learning_rate': 0.0001, 'batch_size': 128}, 80000),
          'ddpg': ({'batch_size': 128, 'buffer_size': 50000, 'learning_rate': 0.001}, 50000),
          'td3': ({'batch_size': 100, 'buffer_size': 1000000, 'learning_rate': 0.001}, 30000)}
tpath = OUT / 'timing.json'
TIMING = json.loads(tpath.read_text()) if tpath.exists() else {}
SEED = 0
for algo, (kw, steps) in PARAMS.items():
    tag = f'{algo}_W0_a_seed{SEED}'
    if tag in TIMING and (OUT / 'models' / f'{tag}.zip').exists():
        print(tag, '이미 완료', TIMING[tag]); continue
    env_train, _ = StockPortfolioEnv(df=train, **env_kwargs).get_sb_env()
    agent = DRLAgent(env=env_train)
    model = agent.get_model(algo, model_kwargs=dict(kw), seed=SEED, verbose=0)
    t0 = time.time(); model = agent.train_model(model=model, tb_log_name=algo, total_timesteps=steps)
    TIMING[tag] = round((time.time() - t0) / 60, 2)
    model.save(str(OUT / 'models' / f'{tag}.zip')); tpath.write_text(json.dumps(TIMING, indent=1))
    print(tag, TIMING[tag], '분')

In [ ]:
# 예산 추정 — v3 9장: 4알고리즘 × 5시드 × 3창 × 3조건
per_seed = sum(TIMING[f'{a}_W0_a_seed0'] for a in PARAMS)
print('시드 1개(4알고리즘) 합계', round(per_seed, 1), '분')
print('W0 전체(5시드 × 3조건)', round(per_seed * 15 / 60, 1), '시간; 롤링 2창 포함 전체', round(per_seed * 45 / 60, 1), '시간(롤링 창은 학습 기간이 짧아 더 적을 수 있음)')
print(TIMING)
print('\n이 셀 출력을 Claude에게 알려 주세요(Drive data/finrl_t3/ 에도 저장됨).')